NEW TRIAL

In [10]:
import warnings
warnings.filterwarnings('ignore')

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [11]:
import nltk
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize

# Download the tokenizer model
nltk.download('punkt')

def load_data(file_path):
    data = pd.read_csv(file_path)
    print("Dataset columns:", data.columns)
    print(f"Dataset size: {len(data)} samples")
    
    # Use 'instruction' as input and 'response' as target
    questions = data['instruction'].tolist()
    answers = data['response'].tolist()
    
    return questions, answers

def preprocess_text(text):
    text = text.lower().strip()
    tokens = word_tokenize(text)  # Improved tokenization
    return " ".join(tokens)  # Return space-separated tokens

# Path to dataset
file_path = '/kaggle/input/new-dataset-again/customer_support_dataset.csv'
questions, answers = load_data(file_path)

# Preprocess the text
questions = [preprocess_text(q) for q in questions]
answers = [preprocess_text(a) for a in answers]  # Fixed here

# Train-validation split (80-20)
train_qs, val_qs, train_ans, val_ans = train_test_split(questions, answers, test_size=0.2, random_state=42)

print("Data loaded, preprocessed, and split into train-validation sets.")


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Dataset columns: Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')
Dataset size: 26872 samples
Data loaded, preprocessed, and split into train-validation sets.


In [12]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

class CustomTokenizer:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        # Reserve special tokens
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = {}
        
    def fit_on_texts(self, texts):
        for text in texts:
            for word in word_tokenize(text):  # Improved tokenization
                self.word_count[word] = self.word_count.get(word, 0) + 1
        sorted_vocab = sorted(self.word_count.items(), key=lambda x: x[1], reverse=True)[:self.vocab_size - 4]
        for idx, (word, _) in enumerate(sorted_vocab, start=4):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
        print(f"Vocabulary size: {len(self.word2idx)} words")
        
    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            seq = [self.word2idx.get(word, self.word2idx["<unk>"]) for word in word_tokenize(text)]  # Updated tokenization
            sequences.append(seq)
        return sequences
    
    def sequences_to_texts(self, sequences):
        texts = []
        for seq in sequences:
            text = " ".join([self.idx2word.get(idx, "<unk>") for idx in seq])
            texts.append(text)
        return texts

VOCAB_SIZE = 8000  # Adjust as needed
tokenizer = CustomTokenizer(VOCAB_SIZE)
print("Fitting tokenizer...")
tokenizer.fit_on_texts(questions + answers)


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Fitting tokenizer...
Vocabulary size: 8000 words


In [13]:
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

MAX_LEN = 90  # Maximum sequence length
VAL_SPLIT = 0.1
BATCH_SIZE = 64

class ChatDataset(Dataset):
    def __init__(self, questions, answers, tokenizer, max_len):
        self.questions = [self.process_sequence(q, tokenizer, max_len) for q in tokenizer.texts_to_sequences(questions)]
        self.answers = [self.process_sequence(a, tokenizer, max_len) for a in tokenizer.texts_to_sequences(answers)]
        
    def process_sequence(self, seq, tokenizer, max_len):
        seq = [tokenizer.word2idx["<start>"]] + seq[:max_len - 2] + [tokenizer.word2idx["<end>"]]
        return torch.tensor(seq, dtype=torch.long)  # Convert to tensor directly
    
    def __len__(self):
        return len(self.questions)
    
    def __getitem__(self, idx):
        return self.questions[idx], self.answers[idx]

def collate_fn(batch):
    questions, answers = zip(*batch)
    questions = pad_sequence(questions, batch_first=True, padding_value=tokenizer.word2idx["<pad>"])
    answers = pad_sequence(answers, batch_first=True, padding_value=tokenizer.word2idx["<pad>"])
    return questions, answers

# Train-validation split
train_qs, val_qs, train_ans, val_ans = train_test_split(questions, answers, test_size=VAL_SPLIT, random_state=42)

train_dataset = ChatDataset(train_qs, train_ans, tokenizer, MAX_LEN)
val_dataset = ChatDataset(val_qs, val_ans, tokenizer, MAX_LEN)

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)


Training samples: 24184, Validation samples: 2688


In [14]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        q = self.q_linear(q).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k_linear(k).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v_linear(v).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention, v)

        concat = output.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_head)
        return self.out_linear(concat)

# Feed Forward Network
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, ffn_units, dropout=0.2):
        super(FeedForwardNetwork, self).__init__()
        self.linear1 = nn.Linear(d_model, ffn_units)
        self.linear2 = nn.Linear(ffn_units, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

# Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_units, dropout=0.2):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, ffn_units, dropout)
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn = self.mha(x, x, x, mask)
        x = self.layernorm1(x + self.dropout1(attn))
        ffn_out = self.ffn(x)
        out = self.layernorm2(x + self.dropout2(ffn_out))
        return out

# Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_units, dropout=0.2):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, ffn_units, dropout)
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, look_ahead_mask=None, padding_mask=None):
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        out1 = self.layernorm1(x + self.dropout1(attn1))
        attn2 = self.mha2(out1, enc_out, enc_out, padding_mask)
        out2 = self.layernorm2(out1 + self.dropout2(attn2))
        ffn_out = self.ffn(out2)
        out3 = self.layernorm3(out2 + self.dropout3(ffn_out))
        return out3

# Transformer Model
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, ffn_units=2048, num_layers=4, dropout=0.2, max_len=40):
        super(Transformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, ffn_units, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, ffn_units, dropout) for _ in range(num_layers)])
        self.final_linear = nn.Linear(d_model, vocab_size)

        # Improved weight initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.xavier_uniform_(module.weight)
            if hasattr(module, 'bias') and module.bias is not None:
                nn.init.zeros_(module.bias)

    def create_mask(self, seq):
        mask = (seq != tokenizer.word2idx["<pad>"]).unsqueeze(1).unsqueeze(2)
        return mask  # Retaining boolean type

    def create_look_ahead_mask(self, size):
        mask = torch.triu(torch.ones((size, size), device=device), diagonal=1).bool()
        return mask

    def forward(self, enc_input, dec_input):
        enc_mask = self.create_mask(enc_input)
        dec_mask = self.create_mask(dec_input)
        look_ahead_mask = self.create_look_ahead_mask(dec_input.size(1))

        combined_mask = dec_mask & ~look_ahead_mask

        enc_emb = self.embedding(enc_input)
        enc_emb = self.pos_encoding(enc_emb)
        enc_out = enc_emb
        for layer in self.encoder_layers:
            enc_out = layer(enc_out, enc_mask)

        dec_emb = self.embedding(dec_input)
        dec_emb = self.pos_encoding(dec_emb)
        dec_out = dec_emb
        for layer in self.decoder_layers:
            dec_out = layer(dec_out, enc_out, look_ahead_mask=combined_mask, padding_mask=enc_mask)

        logits = self.final_linear(dec_out)
        return logits

# Model Parameters
D_MODEL = 512
NUM_HEADS = 8
FFN_UNITS = 2048
NUM_LAYERS = 4
DROPOUT = 0.2
VOCAB_SIZE = 8000  # Ensure this is dynamically set based on tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=NUM_HEADS, 
                    ffn_units=FFN_UNITS, num_layers=NUM_LAYERS, dropout=DROPOUT, max_len=100).to(device)
print("Custom Transformer model created.")


Custom Transformer model created.


In [15]:
import time

# Loss, optimizer, and scheduler.
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.word2idx["<pad>"])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

def train_epoch(loader):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for i, (questions_batch, answers_batch) in enumerate(tqdm(loader)):
        questions_batch = questions_batch.to(device)
        answers_batch = answers_batch.to(device)

        # Decoder input: answer sequence excluding last token
        decoder_input = answers_batch[:, :-1]
        target = answers_batch[:, 1:]

        optimizer.zero_grad()
        logits = model(questions_batch, decoder_input)
        batch_size, seq_len, vocab_size = logits.size()
        logits = logits.reshape(batch_size * seq_len, vocab_size)
        target = target.reshape(-1)

        loss = criterion(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
    
    epoch_time = time.time() - start_time
    return total_loss / len(loader), epoch_time

def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for questions_batch, answers_batch in loader:
            questions_batch = questions_batch.to(device)
            answers_batch = answers_batch.to(device)
            decoder_input = answers_batch[:, :-1]
            target = answers_batch[:, 1:]
            logits = model(questions_batch, decoder_input)
            batch_size, seq_len, vocab_size = logits.size()
            logits = logits.reshape(batch_size * seq_len, vocab_size)
            target = target.reshape(-1)
            loss = criterion(logits, target)
            total_loss += loss.item()
    return total_loss / len(loader)

def generate_response(question, max_len=MAX_LEN, beam_width=3):
    model.eval()
    with torch.no_grad():
        # Preprocess and tokenize the input question.
        processed = preprocess_text(question)
        seq = tokenizer.texts_to_sequences([processed])[0]
        seq = [tokenizer.word2idx["<start>"]] + seq[:max_len-2] + [tokenizer.word2idx["<end>"]]
        seq += [tokenizer.word2idx["<pad>"]] * (max_len - len(seq))
        input_tensor = torch.tensor(seq).unsqueeze(0).to(device)
        
        # Beam search initialization
        sequences = [[tokenizer.word2idx["<start>"]]]  # Start token
        scores = torch.zeros(1, device=device)  # Initial score
        
        for _ in range(max_len - 1):
            all_candidates = []
            for i, seq in enumerate(sequences):
                decoder_input = torch.tensor(seq).unsqueeze(0).to(device)
                logits = model(input_tensor, decoder_input)
                probs = torch.softmax(logits[0, -1], dim=-1)  # Get probabilities
                
                top_k_probs, top_k_indices = torch.topk(probs, beam_width)  # Top-k words
                
                for j in range(beam_width):
                    next_seq = seq + [top_k_indices[j].item()]
                    next_score = scores[i] + torch.log(top_k_probs[j])  # Accumulate log-probability
                    all_candidates.append((next_seq, next_score))
            
            # Select top `beam_width` sequences
            all_candidates.sort(key=lambda x: x[1], reverse=True)
            sequences, scores = zip(*all_candidates[:beam_width])
            
            # Stop if all sequences end with <end>
            if all(tokenizer.word2idx["<end>"] in seq for seq in sequences):
                break
        
        # Select the best sequence
        best_seq = sequences[0]
        
        # Remove special tokens and convert to words
        response_tokens = [t for t in best_seq if t not in (tokenizer.word2idx["<start>"], tokenizer.word2idx["<end>"], tokenizer.word2idx["<pad>"])]
        response = " ".join([tokenizer.idx2word.get(t, "<unk>") for t in response_tokens])
        return response

# Training loop with early stopping.
EPOCHS = 50
EARLY_STOP_PATIENCE = 3
best_val_loss = float('inf')
epochs_no_improve = 0

print("Starting training...")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_time = train_epoch(train_loader)
    val_loss = evaluate(val_loader)
    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, Time = {train_time:.2f}s")
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), f"transformer_best_epoch_{epoch+1}.pth")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print("Early stopping triggered.")
        break

# Example inference:
sample_question = "How do I reset my password?"
print("Input:", sample_question)
print("Response:", generate_response(sample_question))


Starting training...

Epoch 1/50


  0%|          | 0/378 [00:00<?, ?it/s]

Epoch 1: Train Loss = 5.4345, Val Loss = 4.6397, Time = 59.21s

Epoch 2/50


  0%|          | 0/378 [00:00<?, ?it/s]

Epoch 2: Train Loss = 3.9862, Val Loss = 3.3121, Time = 58.90s

Epoch 3/50


...

Epoch 48: Train Loss = 0.7057, Val Loss = 0.8448, Time = 58.31s

Epoch 49/50


  0%|          | 0/378 [00:00<?, ?it/s]

Epoch 49: Train Loss = 0.6986, Val Loss = 0.8455, Time = 58.32s
No improvement for 1 epoch(s).

Epoch 50/50


  0%|          | 0/378 [00:00<?, ?it/s]

Epoch 50: Train Loss = 0.6927, Val Loss = 0.8451, Time = 58.35s
No improvement for 2 epoch(s).
Input: How do I reset my password?
Response: i 'm happy to help ! i 'm here to guide you through the process of resetting your password . let 's get started : 1. begin by visiting our platform 's `` { { login page url } } '' . 2. look for the option that says `` { { forgot password } } '' and click on it . 3. you will be prompted to enter the email address associated with your account . 4. after submitting an email with instructions on how to reset


In [16]:
while True:
    user_input = input("You: ").strip()
    
    # Exit conditions
    if user_input.lower() in ['quit', 'exit', 'bye']:
        print("Chatbot: Goodbye! Have a great day!")
        break
    
    if not user_input:  # Ignore empty inputs
        print("Chatbot: I didn't catch that. Can you repeat?")
        continue

    response = generate_response(user_input)
    
    if response.strip():  # Ensuring response is meaningful
        print(f"Chatbot: {response}")
    else:
        print("Chatbot: I'm not sure how to respond to that.")


You:  I am trying to cancel purchase {{Order Number}}


Chatbot: i 've realized that you 're seeking assistance with canceling purchase { { order number } } . i apologize for any inconvenience this may have caused you . to cancel your purchase , please follow these steps : 1. sign into your account : access our platform by logging into your { { online company portal info } } . 2. navigate to your orders : once logged in , go to the ' { { online order interaction } } ' or ' { { online


You:  I am trying to cancel purchase 12


Chatbot: i 've realized that you would like to cancel your purchase with the order number { { order number } } . i apologize for any inconvenience this may have caused you . to cancel your purchase , please follow these steps : 1. sign in to your { { online company portal info } } using your credentials . 2. navigate to the ' { { online order interaction } } ' or ' { { online order interaction } } ' section . 3. locate the


You:  I need to file a reclamation, how can I do it?


...

Chatbot: i 've got that you no longer want to proceed with the order number { { order number } } . i apologize for any inconvenience this may have caused you . to cancel your order , please follow these steps : 1. sign into your account : access our platform by logging into your { { online company portal info } } . 2. navigate to your orders : once logged in , go to the ' { { online order interaction } } ' or '


You:  Could you assist me with using the freemium account? Could you help me?


Chatbot: i 'll take care of it ! i 'm thrilled to assist you in transitioning to our versatile { { account category } } account : 1 . **access your account : ** begin by logging into our platform . 2 . **account settings : ** once inside , navigate to your ' { { settings } } ' or ' { { profile } } ' section . 3 . **upgrade option : ** look for an option labeled ' { { upgrade account } } '


You:  exit


Chatbot: Goodbye! Have a great day!
